In [1]:
import numpy as np
import pandas as pd
import os
import random
import re


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns



import ete3 as ete

import scipy.stats as stats
from functools import *

import Bio
from Bio import Entrez
from Bio import SeqIO
from Bio import Seq
from Bio.SeqUtils import GC
from Bio.SeqFeature import SeqFeature, FeatureLocation


sns.set_context("paper")
%matplotlib inline

# Key files for this script

This script combines ORF annotations produced from parse_refseq_gbff.ipynb files with things identified from ribosome profiling data. Also compares with Elledge lab human ORF libraries
- refseq_all_orf_df_assigned.csv
- riboseq_orfs_all.csv

Afterwards, this script outputs a file that combines the two files above, removing all duplicated sequences (giving priority in the following ways:

1) all known CDS
2) ribsome profiling sequences
3) unannotated CDSs

The final combined output file is:
- refseq_all_orf_df_assigned_FINAL.csv

In [2]:
outpath = "/lab/solexa_weissman/yhc/viral_smORF/data_analysis/virushostdb/orf_files/"

refseq_path = '/lab/solexa_weissman/yhc/viral_smORF/data_analysis/virushostdb/orf_files/refseq_all_orf_df_assigned.csv'
vhdb_ann = '/lab/solexa_weissman/yhc/viral_smORF/data_analysis/virushostdb/orf_files/virushostdb_orf_df_assigned.csv'
riboseq_ann = '/lab/solexa_weissman/yhc/viral_smORF/data_downloads/viral_riboseq/riboseq_orfs_all.csv'


elledge_ann = '/lab/solexa_weissman/yhc/viral_ORF/vORF_elledge_sequences/ERIC-combined_all_viral9_withpoolid.csv'


In [3]:
Seq.Seq('ATGCCTCTGTCACCAGCTCTTCACGGTGTGCTTACGACTCCTCCTGCGAGAGGGGGGCCTCCTCCAACTCCATATACCAACCGGAAAACAGACGACAAGAGGAAATCAATTCCCCTGAGAGGAGGCCTACCTCGACATCTAACCTTCGACGACGACAAGGAGGAGAACAAGGAGAACCTAGCTCCTCCGGAGGAGCCAAACGACGGAGAGCAGACTCGGGAGGAGGCGCGCCTTCTCCAGAGGAAGTTGGAGAATCTCATCGATCGGTTGAAAGACGACATCTGTCTCGACTTAGAGTACTACAAGAGGAGGCTCGGGACCCACCAA'
).translate()

Seq('MPLSPALHGVLTTPPARGGPPPTPYTNRKTDDKRKSIPLRGGLPRHLTFDDDKE...THQ')

In [4]:
refseq_raw = pd.read_csv(refseq_path, index_col = 0, low_memory=False)
refseq_ann = refseq_raw[refseq_raw.ann]
refseq_un = refseq_raw[~refseq_raw.ann]
display(refseq_raw.head(5))


riboseq= pd.read_csv(riboseq_ann, index_col = None, low_memory=False)
riboseq['ann'] = [True]*len(riboseq)
display(riboseq.head())

refseq = pd.concat([refseq_ann,riboseq, refseq_un], sort=False)
refseq = refseq.drop_duplicates(subset=['seq_aa'], keep='first')
refseq = refseq.sort_values(by = 'length_aa').reset_index(drop=True)
refseq.to_csv(outpath + 'refseq_all_orf_df_assigned_FINAL.csv')


elledge_raw= pd.read_csv(elledge_ann, index_col = 0, low_memory=False)
elledge = elledge_raw[[True if 'IEDB' not in str(e) else False for e in elledge_raw['Taxonomic lineage (SPECIES)'] ]].copy()
display(elledge.head(5))

print(len(refseq_raw))
print(len(riboseq))
print(len(refseq))
print(len(elledge_raw))
print(len(elledge))
print(elledge.columns)
#print(len(df.ORF.unique()))


,virus_name,accession,description,virus_tax_id,genome_type,type,matpro_id,ann,location,product,note,aa_match,seq_nt,seq_aa,length_aa,start_codon
0,NY_014 poxvirus,NC_035469.1,"NY_014 poxvirus strain 2013, complete genome",2025360,dsDNA,CDS,YP_009408449.1,True,[66354:66456](-),Virus entry/fusion complex component,NaN,True,ATGATAGCGGTTGTCATGTTCTTAATAGCATTCATGTTCTGTAGTT...,MIAVVMFLIAFMFCSWLSYSYLRPYINNKPLDN,33,ATG
1,Vaccinia virus,NC_006998.1,"Vaccinia virus, complete genome",10245,dsDNA,CDS,YP_910498.1,True,[59743:59851](-),hypothetical protein,NaN,True,ATGCTCGTCGTAATTATGTTTTTTATAGCGTTTGCCTTCTGTAGTT...,MLVVIMFFIAFAFCSWLSYSYLRPYISTKELNKSR,35,ATG
2,Monkeypox virus,NC_063383.1,"Monkeypox virus, complete genome",10244,dsDNA,CDS,YP_010377183.1,True,[57522:57630](-),"MV membrane, EFC component",NaN,True,ATGCTCGTCGTAATTATGTTTTTTATAGCGTTTGTCTTCTGTAGTT...,MLVVIMFFIAFVFCSWLSYSYLCPYISTKELNKSR,35,ATG
3,Vaccinia virus,NC_006998.1,"Vaccinia virus, complete genome",10245,dsDNA,CDS,YP_910501.1,True,[182355:182469](-),hypothetical protein,hypothetical protein; the poxviridae are envel...,True,ATGTTAAACTTCAGTTTATGTTTGTACCCCGTATTCATACTTAACA...,MLNFSLCLYPVFILNKLVLRTQSIILHTINNASIKNR,37,ATG
4,Vaccinia virus,NC_006998.1,"Vaccinia virus, complete genome",10245,dsDNA,CDS,YP_910500.1,True,[163144:163258](+),hypothetical protein,NaN,True,ATGTCTGGGATAGTAAAATCTATCATATTGAGCGGACCATCTGGTT...,MSGIVKSIILSGPSGLGKTAIAKRLWEYIWICGVPYH,37,ATG


,virus_name,description,accession,type,matpro_id,product,location,strand,polypep?,length_aa,aa_match,start_codon,seq_aa,seq_nt,length_nt,ann
0,Human betaherpesvirus 6B,"Human herpesvirus 6B strain Z29, complete genome",ribo-seq Human betaherpesvirus 6B,CDS,AAD49614.1,DR1,"join{[582:841](+), [954:2975](+)}",1,False,759,True,ATG,MPLTARAGHTLHRLPLSHYWWLLLGRHSLRHVHSYLRLRKGLRLPL...,ATGCCGCTGACGGCGCGTGCCGGCCACACCCTGCATCGTCTTCCGC...,2280,True
1,Human betaherpesvirus 6B,"Human herpesvirus 6B strain Z29, complete genome",ribo-seq Human betaherpesvirus 6B,CDS,AAD49618.1,DR3,[2722:3325](-),-1,False,200,True,ATG,MSRVFSCVLRACVCAGLCCWVCMGVICGDCQRWWRRRCARWGRVGP...,ATGTCTCGCGTGTTCTCGTGCGTGCTCCGCGCGTGTGTATGTGCCG...,603,True
2,Human betaherpesvirus 6B,"Human herpesvirus 6B strain Z29, complete genome",ribo-seq Human betaherpesvirus 6B,CDS,AAD49617.1,B1,[3021:3501](+),1,False,159,True,ATG,MQKNMKTKKTKKRGRKEGNTPETERRMEPARSRTSAIPSGLRRRSG...,ATGCAGAAGAACATGAAGACGAAGAAGACGAAGAAACGAGGACGAA...,480,True
3,Human betaherpesvirus 6B,"Human herpesvirus 6B strain Z29, complete genome",ribo-seq Human betaherpesvirus 6B,CDS,AAD49616.1,B2,[3535:3775](+),1,False,79,True,ATG,MQPMTKKKHTTASKRSPARPPSLSPPLESRRRVGGKSPERLHGNAP...,ATGCAACCGATGACAAAAAAAAAACACACCACCGCAAGCAAACGCT...,240,True
4,Human betaherpesvirus 6B,"Human herpesvirus 6B strain Z29, complete genome",ribo-seq Human betaherpesvirus 6B,CDS,AAD49615.1,DR6,"join{[5026:5330](+), [6328:7203](+)}",1,False,392,True,ATG,MTTRHTQTRDGRIAIRRDGARLAHARARARFEWLLLARGRPSKLYG...,ATGACAACGCGACACACGCAGACGAGGGACGGACGCATCGCGATCC...,1179,True


,Barcode,Batch,Caution,Chain,Chain End,Chain ID,Chain Name,Chain Start,Chain number,Combined concatemer length,...,Twist well number,Version (entry),Version (sequence),Virus host,Weissmann ORF Family,Yield (ng),needs redesign?,Subpool,Superpool,ORF position
1444,gttacgctatacgtttactccgga,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,C7,NaN,NaN,NaN,NaN,351,False,[51],['Q'],1
1445,tccccattggtaccgacctattgt,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,G10,NaN,NaN,NaN,NaN,149,False,[10],['B'],1
2196,agggacgttgaagtgcccccgtgc,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,D10,NaN,NaN,NaN,NaN,269,False,[53],['R'],1
2236,gttatgggtccaaaacaaccgaaa,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,537.0,...,D1,NaN,NaN,NaN,NaN,249,False,[53],['R'],1
3132,ggggtgcttaccgcaactaagccc,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,537.0,...,D1,NaN,NaN,NaN,NaN,333,False,[12],['C'],1


451916
1766
452526
13428
12651
Index(['Barcode', 'Batch', 'Caution', 'Chain', 'Chain End', 'Chain ID',
       'Chain Name', 'Chain Start', 'Chain number',
       'Combined concatemer length', 'Concatemer',
       'Concatemer fragment lengths', 'Concatemer order',
       'Cross-reference (EMBL)', 'Cross-reference (RefSeq)',
       'Cross-reference (UniGene)', 'EMBL_ids_parsed',
       'Endogenous nucleotide seuquence', 'Entry', 'Entry name',
       'Final insert sequence', 'Fragment_ID', 'Fragment end',
       'Fragment length', 'Fragment nucleotide end',
       'Fragment nucleotide length', 'Fragment nucleotide sequence',
       'Fragment nucleotide sequence (GC low)',
       'Fragment nucleotide sequence (no ATG)',
       'Fragment nucleotide sequence (rare codons and retriction sites removed with adaptors)',
       'Fragment nucleotide sequence (rare codons and retriction sites removed)',
       'Fragment nucleotide sequence (rare codons removed)',
       'Fragment nucleotide start',

In [5]:
print(len(elledge))
elledge.head().columns

12651


Index(['Barcode', 'Batch', 'Caution', 'Chain', 'Chain End', 'Chain ID',
       'Chain Name', 'Chain Start', 'Chain number',
       'Combined concatemer length', 'Concatemer',
       'Concatemer fragment lengths', 'Concatemer order',
       'Cross-reference (EMBL)', 'Cross-reference (RefSeq)',
       'Cross-reference (UniGene)', 'EMBL_ids_parsed',
       'Endogenous nucleotide seuquence', 'Entry', 'Entry name',
       'Final insert sequence', 'Fragment_ID', 'Fragment end',
       'Fragment length', 'Fragment nucleotide end',
       'Fragment nucleotide length', 'Fragment nucleotide sequence',
       'Fragment nucleotide sequence (GC low)',
       'Fragment nucleotide sequence (no ATG)',
       'Fragment nucleotide sequence (rare codons and retriction sites removed with adaptors)',
       'Fragment nucleotide sequence (rare codons and retriction sites removed)',
       'Fragment nucleotide sequence (rare codons removed)',
       'Fragment nucleotide start', 'Fragment sequence',
       'F

In [6]:
print('Number of unique organisms in Elledge ORF library')
elledge['virus_tax_id'] = np.array(elledge['Organism ID'], dtype = float)
print(elledge['virus_tax_id'].nunique())


by_taxid = elledge['virus_tax_id'].isin(list(refseq.virus_tax_id))
by_org = elledge['Organism'].isin(list(refseq.virus_name))
by_matpro_id = elledge['NCBI ID'].isin(list(refseq['matpro_id']))
by_seq_aa = elledge['ORF sequence'].isin(refseq.seq_aa)

elledge['Human?'] = np.array([by_matpro_id | by_taxid])[0]
elledge_human = elledge[elledge['Human?']].reset_index(drop = True)
elledge_nonhuman = elledge[~elledge['Human?']].reset_index(drop = True)
print('Number of organisms with human hosts in Elledge library')
print(elledge_human['Organism'].nunique())


print('Number of organisms in my analysis')
refseq.virus_tax_id.nunique()


Number of unique organisms in Elledge ORF library
716
Number of organisms with human hosts in Elledge library
533
Number of organisms in my analysis


502

In [7]:
#I suspect the ones that are missing are things that they chopped up into pieces
#seems like they're all things that were chopped up or are records that have since been updated or removed by ncbi
missing = elledge_human[~elledge_human['ORF sequence'].isin(list(refseq.seq_aa)) & 
                        ~elledge_human['Full original entry sequence'].isin(list(refseq.seq_aa)) & 
                        ~by_matpro_id & 
                        ~elledge_human['Fragment nucleotide sequence'].isin(list(refseq.seq_nt))
                       ].sort_values('ORF length')
missing = missing[~missing['ORF sequence'].duplicated()].reset_index()
print(elledge_human['ORF sequence'].nunique())
print(missing['ORF sequence'].nunique())
print(missing[missing['ORF length']<106]['ORF sequence'].nunique())

missing.to_csv(outpath+'elledge_missing.csv')

7651
591
88


/tmp/ipykernel_4101510/3571364334.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  missing = elledge_human[~elledge_human['ORF sequence'].isin(list(refseq.seq_aa)) &


In [8]:
missing[missing['ORF length']<106]

,index,Barcode,Batch,Caution,Chain,Chain End,Chain ID,Chain Name,Chain Start,Chain number,...,Version (sequence),Virus host,Weissmann ORF Family,Yield (ng),needs redesign?,Subpool,Superpool,ORF position,virus_tax_id,Human?
0,5619,atgcccacttgcgtgttagacagg,3,NaN,CHAIN 2 584 Gag-Pro polyprotein. {ECO:0000250}...,584.0,{ECO:0000250}. /FTId=PRO_0000259821.,Transframe peptide,562.0,5.0,...,3.0,NaN,NaN,371,False,"[17, 56]","['E', 'S']",2,402036.0,True
1,2285,atgaatgcctgcaattacccggtg,3,NaN,CHAIN 1 2332 Genome polyprotein. /FTId=PRO_000...,1625.0,{ECO:0000255}. /FTId=PRO_0000039826.;,Protein 3B-2,1602.0,12.0,...,NaN,Bos taurus (Bovine) [TaxID: 9913]; Capra hircu...,NaN,NaN,True,Nan,Nan,Nan,12112.0,True
2,9732,tgtgctgatgaacaatcttcctgc,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,ORFL240C,NaN,True,Nan,Nan,Nan,295027.0,True
3,9736,aatcgacgttgttcaaaggataca,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,ORFS362W,NaN,True,Nan,Nan,Nan,295027.0,True
4,5635,acaaatagccggcatccg,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Homo sapiens (Human) [TaxID: 9606],NaN,NaN,True,Nan,Nan,Nan,67605.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,3754,gccatctgcaagctcaagaattac,3,NaN,CHAIN 1 100 Uncharacterized protein UL64. /FTI...,100.0,/FTId=PRO_0000115334.,Uncharacterized protein UL64,1.0,0.0,...,1.0,NaN,NaN,NaN,True,Nan,Nan,Nan,10360.0,True
84,1607,gcacatctatcgtcatgt,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Culex nigripalpus [TaxID: 42429],NaN,NaN,True,Nan,Nan,Nan,645993.0,True
85,8830,ttgcgtataaaacagaga,2,NaN,CHAIN 1 103 Uncharacterized protein C20L. /FTI...,103.0,/FTId=PRO_0000099420.,Uncharacterized protein C20L,1.0,0.0,...,1.0,NaN,NaN,280,False,[47],['O'],2,10249.0,True
86,5899,tattacattcatcgagac,2,NaN,CHAIN 1 104 Uncharacterized protein 108L. /FTI...,104.0,/FTId=PRO_0000377812.,Uncharacterized protein 108L,1.0,0.0,...,NaN,Aedes vexans (Inland floodwater mosquito) (Cul...,NaN,NaN,True,Nan,Nan,Nan,345201.0,True


In [9]:
missing[~missing['ORF sequence'].duplicated()].reset_index(drop=True)

,index,Barcode,Batch,Caution,Chain,Chain End,Chain ID,Chain Name,Chain Start,Chain number,...,Version (sequence),Virus host,Weissmann ORF Family,Yield (ng),needs redesign?,Subpool,Superpool,ORF position,virus_tax_id,Human?
0,5619,atgcccacttgcgtgttagacagg,3,NaN,CHAIN 2 584 Gag-Pro polyprotein. {ECO:0000250}...,584.0,{ECO:0000250}. /FTId=PRO_0000259821.,Transframe peptide,562.0,5.0,...,3.0,NaN,NaN,371,False,"[17, 56]","['E', 'S']",2,402036.0,True
1,2285,atgaatgcctgcaattacccggtg,3,NaN,CHAIN 1 2332 Genome polyprotein. /FTId=PRO_000...,1625.0,{ECO:0000255}. /FTId=PRO_0000039826.;,Protein 3B-2,1602.0,12.0,...,NaN,Bos taurus (Bovine) [TaxID: 9913]; Capra hircu...,NaN,NaN,True,Nan,Nan,Nan,12112.0,True
2,9732,tgtgctgatgaacaatcttcctgc,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,ORFL240C,NaN,True,Nan,Nan,Nan,295027.0,True
3,9736,aatcgacgttgttcaaaggataca,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,ORFS362W,NaN,True,Nan,Nan,Nan,295027.0,True
4,5635,acaaatagccggcatccg,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Homo sapiens (Human) [TaxID: 9606],NaN,NaN,True,Nan,Nan,Nan,67605.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,2220,ccgaagtatcctgcaatcagaaag,3,NaN,CHAIN 1 3755 Large tegument protein deneddylas...,3755.0,/FTId=PRO_0000406058.,Large tegument protein deneddylase,1.0,0.0,...,NaN,Equus caballus (Horse) [TaxID: 9796],NaN,NaN,False,Nan,['23'],1,82831.0,True
587,6936,aacccgatcctgcgaaaagacagt,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,300,False,"[16, 50]","['D', 'Q']",1,1562066.0,True
588,8302,ccgcagcctactaacaatgcagaa,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,250,False,"[4, 43]","['A', 'O']",1,1495316.0,True
589,3345,gtagatactgctggggaagagcag,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,False,Nan,['37'],1,1298362.0,True


In [10]:
min(elledge['Fragment nucleotide length'])

24